## 1. Load and Import

In [23]:
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, MiniMap
from pathlib import Path
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DATA_PROC = Path("../data/processed")
DATA_EXT  = Path("../data/external")
MAPS_OUT  = Path("../outputs/maps")
MAPS_OUT.mkdir(parents=True, exist_ok=True)

## 2. Load tracts

In [24]:
tracts = gpd.read_file(DATA_PROC / "tract_features_predicted.geojson")
tracts = tracts.to_crs("EPSG:4326")  # Folium needs WGS84

stations  = gpd.read_file(DATA_PROC / "all_stations.geojson").to_crs("EPSG:4326")
hospitals = gpd.read_file(DATA_PROC / "hospitals.geojson").to_crs("EPSG:4326")
hydrants  = gpd.read_file(DATA_PROC / "Fire_Hydrants.geojson").to_crs("EPSG:4326")

# Load road graph for routing
G = ox.load_graphml(DATA_EXT / "road_network.graphml")

print(f"Tracts: {len(tracts)}  |  Stations: {len(stations)}  |  Hospitals: {len(hospitals)}")

Tracts: 2493  |  Stations: 412  |  Hospitals: 165


## 3. Color Scheme

In [25]:
TIER_COLORS = {
    "Low":      "#2ecc71",   # green
    "Moderate": "#f1c40f",   # yellow
    "High":     "#e67e22",   # orange
    "Critical": "#e74c3c",   # red
}

def tier_color(tier_label):
    return TIER_COLORS.get(tier_label, "#cccccc")

## 4. Risk Map

In [26]:
la_center = [34.0522, -118.2437]

m = folium.Map(location=la_center, zoom_start=10,
               tiles="CartoDB positron")

# Add tract choropleth
for _, row in tracts.iterrows():
    folium.GeoJson(
        row["geometry"].__geo_interface__,
        style_function=lambda feat, color=tier_color(row["predicted_tier_label"]): {
            "fillColor": color,
            "color": "white",
            "weight": 0.4,
            "fillOpacity": 0.6,
        },
        tooltip=folium.Tooltip(
            f"Tract: {row['tract']}<br>"
            f"Risk: <b>{row['predicted_tier_label']}</b><br>"
            f"SVI: {row['rpl_themes']:.2f}<br>"
            f"Dist → Station: {row['dist_fire_station_m']:.0f}m<br>"
            f"Dist → Hospital: {row['dist_hospital_m']:.0f}m"
        )
    ).add_to(m)

MiniMap().add_to(m)
m.save(MAPS_OUT / "01_risk_choropleth.html")
print("Saved 01_risk_choropleth.html")

Saved 01_risk_choropleth.html


## 5. Add fire stations and hospitals

In [27]:
# Fire stations layer
station_group = folium.FeatureGroup(name="Fire Stations")
for _, row in stations.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color="#1a1a2e",
        fill=True,
        fill_color="#3498db",
        fill_opacity=0.9,
        tooltip=row.get("STATION_NAME", "Fire Station")
    ).add_to(station_group)
station_group.add_to(m)

# Hospitals layer
hospital_group = folium.FeatureGroup(name="Hospitals")
for _, row in hospitals.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        icon=folium.Icon(color="red", icon="plus", prefix="fa"),
        tooltip=row.get("name", "Hospital")
    ).add_to(hospital_group)
hospital_group.add_to(m)

folium.LayerControl().add_to(m)
m.save(MAPS_OUT / "02_risk_with_infrastructure.html")
print("Saved 02_risk_with_infrastructure.html")

Saved 02_risk_with_infrastructure.html
